<a href="https://colab.research.google.com/github/perarneskjelvik/Selvstudium-KI/blob/main/Lungebetennelse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# OPPDAGE LUNGEBETENNELSE FRA RØNTGENBILDER
# BRUK AV CNN
#
# 1. NEDLASTING AV DATA FRA KAGGLE (røntgenbilder)
#
# Før koden i denne cellen kjøres må:
# - Kaggle>Profil>Settings>Create Legacy API Key
# - filen kaggle.json genereres fra Kaggle
#   inneholder brukernavn og API key som gir tilgang til Kaggle

# - filen kaggle.json må lastes opp til root i colab
#   (filer symbolet i venstre meny)
# Koden i denne cellen laster ned røntgenbildene som skal
# brukes i maskinlæringsmodellen fra Kaggle.com.
#
# Det lages en katalog "chest_xray" med 3 underkataloger
# "test", "train" og "val". Alle disse 3 katalogene har 2
# underkataloger "NORMAL" og "PNEUMONIA". Disse katalogene inneholder
# røntgenbildene (.jpg filer).
#
# installerer bibliotek som gjør det mulig å kommunisere med Kaggle.com
!pip install kaggle

# lager katalog og kopierer filen kaggle.json dit og setter filrettighetene
# til read og write (600)
!mkdir /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

# bruker Kaggle bobliotekets kommandolinje verktøy og laster ned
# datasettet paultimothymooney/chest-xray-pneumonia som inneholder
# røntgenbilder av lunger
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

# pakker ut bildene og sletter to unødvendige kataloger som er laget
!unzip chest-xray-pneumonia.zip
!rm -rf chest_xray/__MACOSX
!rm -rf chest_xray/chest_xray

# Har nå lagret bildene i en katalogstruktur:
# content>chest_xray
#     test
#        NORMAL
#           *.jpg bilder av normale lunger
#        PNEUMONIA
#           *.jpg bilder av lunger med lungebetennelse
#     train
#        NORMAL
#        PNEUMINIA
#     val
#        NORMAL
#        PNEUMONIA

In [ ]:
# 2. TILPASNING AV DATA (TRAIN/TEST)
#
# Bildene er ikke fordelt optimalt i train/validate/test.
# Å dele data inn i trenings-, validerings- og testsett er en fundamental
# praksis i maskinlæring for å sikre at modellen lærer generelle mønstre,
# i stedet for bare å huske treningsdataene (overfitting).
#yyyy
# 1. Treningssett (Train Set)
# Formål: Brukes til å trene modellen. Algoritmen ser på disse dataene for å
# finne mønstre, justere vekter og lære sammenhenger mellom input og ønsket
# output. Dette er det største datasettet, ofte ca. 60–80 % av totalen.
#
# 2. Valideringssett (Validation Set)
# Formål: Brukes underveis i treningsprosessen for å justere hyperparametre
# og evaluere modellen mens den trenes. Det fungerer som en kontroll for å se
# om modellen overfitter (blir for spesialisert på treningsdataene).
# Modellen trenes ikke direkte på disse dataene, men de brukes til å
# velge hvilken versjon av modellen som fungerer best (f.eks. hvilken "epoch"
# eller læringsrate).
#
# 3. Testsett (Test Set)
# Formål: Brukes kun én gang, til slutt, for å gi en upartisk evaluering av
# den ferdigtrente modellen. Det forteller deg hvor godt modellen vil fungere
# på helt nye, usette data.
# Ingen justeringer av modellen skal gjøres basert på testsettet.
# Hvis du endrer modellen etter å ha sett testresultatene, blir testdataene
# en del av treningsprosessen, og evalueringen blir upålitelig.
#
# En vanlig fordeling av det totale datagrunnlaget er:
# 60% Train / 20% Validation / 20% Test
# Eller 70/15/15 for større datasett.
# Uten denne oppdelingen risikerer man at modellen ser ut til å fungere
# fantastisk på data den allerede har sett, men feiler fullstendig i praktisk
# bruk.

# Fordeler bildene på en bedre måte
# (dvs 80% (train/validate), 20% av bildene til test).
# Bildene flyttes ikke, det lages
# - en dataframe for train med filsti/label
# - en dataframe for test med filsti/label.
# Skriver ut ny fordeling slik at det kan bekreftes ok

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# imutils er et bibliotek for å håndtere bilder
# modulen paths finner alle bildene i en katalog
from imutils import paths

from sklearn.model_selection import train_test_split
# train_test_split() funksjonen er viktig i maskinlæring for å splitte
# et datasett i to nye datasett - training og testing.

def generate_dataframe(directory):
  # Funksjonen lager en pandas dataframe med 2 kolonner.
  # kolonne 1: filsti til bildene
  # kolonne 2: label (status/fasit) til bildene (normal eller pneumonia)

  # Bruker paths.list_images() for å hente bildenes filstier rekursivt
  # Resultatet er en generator, den konverteres til en liste
  img_paths = list(paths.list_images(directory))

  # labels = ['normal' if x.find('NORMAL') > -1 else 'pn' for x in img_paths]

  # Lager en liste labels som inneholder status til bildene.
  # Går gjennom alle filstiene. Inneholder de "NORMAL" er status normal,
  # hvis ikke er status pn (lungebetennelse.)
  labels = []
  for x in img_paths:
    if x.find('NORMAL') > -1:
      labels.append('normal')
    else:
      labels.append('pn')

  return pd.DataFrame({'paths': img_paths, 'labels': labels})

# lager dataframe med alle filstiene og labels
all_df = generate_dataframe('chest_xray')

# splitter filstiene og labels i to datasett: train/validate og test
# 20% av de totale filstiene/labels går til test
train, test = train_test_split(all_df, test_size=0.2, random_state=42)

# skriver ut antall labels i datasettene train og test.
# ser at test er 20% og at fordelingen av normal og pn er lik i train og test.
print(train['labels'].value_counts())
print(test['labels'].value_counts())

# skriver ut plot (stolpediagram) over labels i de to settene
fig = plt.figure(figsize=(10, 3))
for idx, x in enumerate([train, test]):
  fig.add_subplot(1,2,idx+1)
  x['labels'].value_counts().plot(kind='bar')


In [ ]:
# 3. LAGE DATA GENERATORER
#
# Keras ImageDataGenerator er en klasse som brukes for real-time
# data augmentation og batch generering av bilder.
# Klassen gjør tilfeldige endringer på treningsbilder, som hjelper mot
# overfitting og forbedrer modellens evne til generalisering.


from tensorflow.keras.preprocessing.image import ImageDataGenerator

def create_generators(train, test, size=224, b=32):
  # Funksjonen lager 3 generatorer: train, validation og test
  # Inndata er train og test dataframe, size brukes for skalering og
  # b er batch størrelsen for hver epoch.
  #

  # lager et ImageDataGenerator objekt for å splitte i train og validate (20%),
  # og lage tilfeldige endringer på bildene
  train_generator = ImageDataGenerator(rescale=1./255, rotation_range=5,
                    width_shift_range=0.1, height_shift_range=0.1,
                    validation_split=0.2)

  # lager et ImageDataGenerator objekt for test,
  # bildene endres ikke, de bare skaleres
  test_generator = ImageDataGenerator(rescale=1./255)

  # dict med inndata
  baseargs = {"x_col": 'paths',
              "y_col": 'labels',
              "class_labels": ['normal', 'pn'],
              "class_mode": 'binary',
              "target_size": (size,size),
              "batch_size": b,
              "seed": 42 }

  # fyller train generatoren med bilder
  train_generator_flow = train_generator.flow_from_dataframe(
    **baseargs, dataframe=train, subset='training')

  # fyller validate generatoren med bilder
  validation_generator_flow = train_generator.flow_from_dataframe(
    **baseargs, dataframe=train, subset='validation')

  # fyller test generatoren med bilder
  test_generator_flow = test_generator.flow_from_dataframe(
      **baseargs, dataframe=test, shuffle=False)

  return train_generator_flow, validation_generator_flow, test_generator_flow

#tester om dette virker
train_generator, validation_generator, test_generator = create_generators(train, test, 224, 32)

print("TRAIN")
print("")
imgs = next(train_generator)
fig = plt.figure(figsize=(10,10))
for i in range(16):
  fig.add_subplot(4,4,i+1)
  image = imgs[0][i]
  label = 'PNEUMONIA' if imgs[1][i] == 1 else 'NORMAL'
  plt.imshow(image)
  plt.title(label)

print("VALIDATE")
print("")
imgs = next(validation_generator)
fig = plt.figure(figsize=(10,10))
for i in range(16):
  fig.add_subplot(4,4,i+1)
  image = imgs[0][i]
  label = 'PNEUMONIA' if imgs[1][i] == 1 else 'NORMAL'
  plt.imshow(image)
  plt.title(label)

print("TEST")
print("")
imgs = next(test_generator)
fig = plt.figure(figsize=(10,10))
for i in range(16):
  fig.add_subplot(4,4,i+1)
  image = imgs[0][i]
  label = 'PNEUMONIA' if imgs[1][i] == 1 else 'NORMAL'
  plt.imshow(image)
  plt.title(label)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Conv2D, MaxPooling2D, Flatten, Dropout, Input
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, TensorBoard, EarlyStopping
from tensorflow.keras.metrics import Recall, Precision, AUC
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.utils import plot_model
from datetime import datetime

def make_smallnet():
  SIZE = 224
  model = Sequential()

  # Corrected line: Use Input(shape) for the first layer
  model.add(Input(shape=(SIZE, SIZE, 3))) # Explicitly define the input layer
  model.add(Conv2D(32, (3, 3), activation="relu")) # Removed input_shape from Conv2D

  model.add(MaxPooling2D((2, 2)))
  model.add(Conv2D(32, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2)))
  model.add(Conv2D(32, (3, 3), activation="relu"))
  model.add(Flatten())
  model.add(Dense(32, activation="relu"))
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer=Adam(learning_rate=0.0001),
                loss='binary_crossentropy',
                metrics=['accuracy',
                         Recall(name='recall'),
                         Precision(name='precision'),
                         AUC(name='auc')])

  return model

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Conv2D, MaxPooling2D, Flatten, Dropout, Input
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, TensorBoard, EarlyStopping
from tensorflow.keras.metrics import Recall, Precision, AUC
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.utils import plot_model
from datetime import datetime

def get_callbacks(model_name):
  # Definerer metoden "get_callbacks" som returnerer en liste av callbacks
  # som brukes senere i treningen.

  # Argumentet "model_name" er modellen som hjelper oss til å lagre
  # filer som er knyttet til modellen.

  # Første callback er TensorBoard.
  # Det er et visualiseringsverktøy som viser hvordan treningen går ved å vise/
  # plotte målinger som vi definerer.
  # Tensorboard kan lastes både før og etter treningen med følgende:
  # %load_ext tensorboard
  # %tensorboard --logdir logs
  # Tensorboard leser fra en katalog som spesifiseres med --logdir FOLDERNAME
  # og leter etter tensorboard log filer.
  # Tensorboard log filer genereres kun hvis tensorboard callback sendes
  # til nettvereket når du setter det opp til trening.
  # Log filene inneholder all målingene ved hvert tidspunkt under
  # trening og validering.
  # Tensorboard plotter disse målingene.

  # Vi trenger kun å spesifisere filsti til logfilene.
  # For å unngå overskriving hvis vi trener netteverket flere ganger
  # inkuderer vi tidspunkt i filstien.

  logdir = (
      f'logs/scalars/{model_name}_{datetime.now().strftime("%m%d%Y-%H%M%S")}'
  )

  tb = TensorBoard(log_dir=logdir)

  # Andre callback er EarlyStopping.
  # Det stopper treningen av nettverket tidlig hvis det er nødvendig.
  # Vi ønsker å unngå overfitting.
  # Hvis validation loss ikke bedres med over 1% i løpet av 20 epochs, er
  # det sannsynligvos trent nok.

  # Argumentet  monitor="val_loss" ber callback om å sjekke validation loss.
  # Argumentet min_delta=1 ber callback sjekke om det har vært mindre enn
  # 1% endring (nedover pga mode =min) etter patience=20 epochs.
  # Argumentet verbose=2 angis for å få print når nettverket stanses tidlig.
  # Til slutt settes modellen tilbake til beste status restore_best_weights=True.


  es = EarlyStopping(
        monitor="val_loss",
        min_delta=0.001, # Changed from 1 to 0.001 for more sensitive stopping
        patience=5, # Increased patience slightly to allow for minor fluctuations
        verbose=2,
        mode="min",
        restore_best_weights=True,
    )

  # Tredje callback er ModelCheckpoint.
  # Den lagrer (hos colab) en versjon av modellen ved hver epoch.
  # Første argument er modellens filnavn.
  # Andre argument er save_best_only=True, som betyr at det modellen lagres kun
  # hvis den er bedre enn forrige modell som ble lagret.
  # Hav avgjør beste modell?
  # Vi ser på monitor='val_loss' og velger modellen som er lavest mode='min'
  # Når vi er ferdige kan vi laste ned modellen (model_smallnet.hdf5),
  # og laste den opp til colab/run igjen eller kjøre den lokalt.

  mc = ModelCheckpoint(f'model_{model_name}.keras',
                      save_best_only=True,
                      verbose=0,
                      monitor='val_loss',
                      mode='min')

  # Siste callback er ReduceLROnPlateau.
  # Generelt - en for hæy læringsrate kan medføre at nettverket ikke konvergerer.
  # Vanligvis vil adam optimizer selv tilpasse læringsraten , men noen
  # ganger må adam ha en tvungen senkning i læringsraten for å konvergere videre.
  # I dette tilfellet reduseres læringsraten med factor=0.3 hvis Validation loss
  # ikke er endret i løpet av 3 epochs (monitor='val_loss', patience=3).
  # Dette fortsetter til læringsratene er 0.000001 (min_lr=0.000001).
  # Den vil også printe når den senker læringsraten (verbose=1).

  rlr = ReduceLROnPlateau(monitor='val_loss',
                          factor=0.3,
                          patience=2,
                          min_lr=0.000001,
                          verbose=1)
  return [tb, es, mc, rlr]

In [11]:
# 6. DEFINERE TRENINGSMETODE OG TRENING AV MODELLEN
#

# Vi definerer en metode som kaller metoden make_smallnet,
# kaller metoden get_callbacks og trener nettverket.

# Metoden tar ca 30 minutter å kjøre!


#epochs = 30
#batch_size = 16  # evt. 8 hvis mulig
#dropout = 0.35–0.4


def fit_model(train_generator, validation_generator, model_name,
              batch_size=8, epochs=15):

  # Inndata er train og validation generatorene vi har laget.
  # Disse brukes i treningen.
  # Modellen trenes med bildene fra training generator images,
  # modellen evalueres hved hver epoch (runde)  med bilder fra
  # validation generator.

  if model_name == 'smallnet':
    model = make_smallnet()

  # Printer oppsummering av modellen.
  model.summary()

  # Skriver ut en figur/plot av modellens lag.
  plot_model(model, to_file=model_name + '.jpg', show_shapes=True)

  # Trener modellen.
  # Inndata er training generator, validation generator,
  # Antall steg for training and validation:
  # Ett steg (step) er en gjennomgang av en batch med bilder.
  # Alle stegene blir antall batches ?
  # Vi finner dette med å kalle metoden generator_name.n
  # (gir oss totalt antall bilder) og deler det på batch size.
  # steps_per_epoch er lik antall batches i training generator
  # og validations_steps er lik antall batches i validation generator.
  # Verbose=1 gir oss oppdatering om fremgang under trening
  # Setter callbacks til get_callbacks()

  print(train_generator.n)
  print(batch_size)
  print(int(train_generator.n/batch_size))
  input()

  model_history = model.fit(train_generator,
                            validation_data=validation_generator,
                           # steps_per_epoch=int(train_generator.n/batch_size),
                          # validation_steps=int(validation_generator.n/batch_size),
                            epochs=epochs,
                            verbose=1,
                            callbacks=get_callbacks(model_name))

# Returnerer ferdig trent modell, historie

# Lagrer modellen
  model.save('test.keras')

  return model, model_history

In [ ]:
# 7. TRENING AV MODELLEN (IKKE TESTING)
#
# Kjøring av metoden fit_model tar ca 30 minutes.

# Trening av modellen.
small_model, small_model_hist = fit_model(train_generator,
                                          validation_generator,
                                          'smallnet', epochs=15)



In [ ]:
# 8. TESTING AV MODELLEN
#
#


import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array


from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
import numpy as np

def predict_image(model, img_path):
  img = img_to_array(load_img(img_path, target_size=(224,224,3)))
  img = img * (1./255)
  img = np.expand_dims(img, axis=0)
  pred = model.predict(img)
  label = 'LUNGEBETENNELSE' if pred >= 0.5 else 'NORMAL'
  print("Diagnose: ", label, "P(Lungebetennelse): ", pred[0][0])

def predict_and_show_directory(model, image_dir, fasit):
    image_files = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    for img_name in image_files:
        img_path = os.path.join(image_dir, img_name)

        # --- samme kode som i predict_image ---
        img = img_to_array(load_img(img_path, target_size=(224,224,3)))
        img = img * (1./255)
        img = np.expand_dims(img, axis=0)

        pred = model.predict(img, verbose=0)
        prob = pred[0][0]
        label = 'LUNGEBETENNELSE' if prob >= 0.5 else 'NORMAL'

        # --- vis bilde ---
        plt.figure(figsize=(4, 4))
        plt.imshow(load_img(img_path))
        plt.axis("off")
        #plt.title(f"{label} (Lungebetennelse = {prob:.2f})")
        plt.title(f"{img_path}")
        plt.show()

        print(f"Filnavn: {img_path}")
        print(f"Fasit: {fasit}")
        print(f"Diagnose: {label}")
        print(f"P(Lungebetennelse) = {prob:.4f}")

#predict_image(small_model, "chest_xray/test/NORMAL/IM-0001-0001.jpeg")
print("Tester normale lunger:")
predict_and_show_directory(small_model, "chest_xray/test/NORMAL/", "NORMAL")
print("Tester lunger med lungebetennelse:")
predict_and_show_directory(small_model, "chest_xray/test/PNEUMONIA/", "LUNGEBETENNELSE")




